In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json, random, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/drive/MyDrive/lane detection"
DATA_DIR = f"{BASE_DIR}/data"
CULANE_DRIVE_DIR = f"{DATA_DIR}/culane/CULane"
BDD_DIR = f"{DATA_DIR}/bdd100k"
BDD_IMAGES_DIR = f"{BDD_DIR}/bdd100k/bdd100k/images/100k/val"
BDD_LABELS = f"{BDD_DIR}/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json"
PROCESSED_DIR = f"{DATA_DIR}/processed"
SPLITS_DIR = f"{BASE_DIR}/splits"
CULANE_DIR = "/content/culane"
LOCAL_PROCESSED = "/content/processed"
for d in [PROCESSED_DIR, SPLITS_DIR, CULANE_DIR, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

W, H = 640, 224
DS = {'culane': dict(w=1640, h=590, crop=236), 'bdd': dict(w=1280, h=720, crop=302)}
N_TRAIN, N_VAL, N_TEST_CULANE, N_TEST_BDD = 45000, 9500, None, None

In [3]:
print(os.listdir(CULANE_DRIVE_DIR))
print(os.listdir(BDD_DIR))
!find "{BDD_DIR}" -maxdepth 6 -name "*.json"
!find "{BDD_DIR}" -maxdepth 6 -type d -name "val"
!find "{BDD_DIR}" -maxdepth 6 -name "*.mov" | head -3
print(os.path.exists(BDD_IMAGES_DIR), os.path.exists(BDD_LABELS))
!df -h /content

['driver_37_30frame.tar.gz', 'driver_100_30frame.tar.gz', 'driver_193_90frame.tar.gz', 'laneseg_label_w16.tar.gz', 'laneseg_label_w16_test.zip', 'annotations_new.tar.gz', 'driver_161_90frame.tar.gz', 'driver_182_30frame.tar.gz', 'driver_23_30frame.tar.gz', 'list.tar.gz', 'video_example.zip']
['archive.zip']
False False
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   88M  108G   1% /


In [4]:
for f in sorted(os.listdir(CULANE_DRIVE_DIR), key=lambda f: ('annotations_new' in f, f)):
    flag = f"{CULANE_DIR}/.{f}.done"
    if f.endswith(('.tar.gz', '.tar')) and not os.path.exists(flag):
        print('extracting', f)
        subprocess.run(['tar', '-xf', f"{CULANE_DRIVE_DIR}/{f}", '-C', CULANE_DIR], check=True)
        open(flag, 'w').close()
if os.path.exists(f"{CULANE_DIR}/annotations_new"):
    shutil.copytree(f"{CULANE_DIR}/annotations_new", CULANE_DIR, dirs_exist_ok=True)
print(os.listdir(CULANE_DIR))

extracting driver_100_30frame.tar.gz
extracting driver_161_90frame.tar.gz
extracting driver_182_30frame.tar.gz
extracting driver_193_90frame.tar.gz
extracting driver_23_30frame.tar.gz
extracting driver_37_30frame.tar.gz
extracting laneseg_label_w16.tar.gz
extracting list.tar.gz
extracting annotations_new.tar.gz
['.driver_37_30frame.tar.gz.done', 'driver_182_30frame', '.driver_193_90frame.tar.gz.done', 'list', '.driver_100_30frame.tar.gz.done', 'driver_193_90frame', 'driver_100_30frame', 'driver_37_30frame', '.driver_161_90frame.tar.gz.done', 'driver_161_90frame', '.driver_23_30frame.tar.gz.done', '.annotations_new.tar.gz.done', '.laneseg_label_w16.tar.gz.done', '.driver_182_30frame.tar.gz.done', '.list.tar.gz.done', 'laneseg_label_w16', 'driver_23_30frame']


In [5]:
import zipfile

BDD_ZIP = f"{DATA_DIR}/bdd100k/archive.zip"
BDD_LOCAL = "/content/bdd"

with zipfile.ZipFile(BDD_ZIP) as z:
    members = [n for n in z.namelist() if ('images/100k/val/' in n and n.endswith('.jpg')) or n.endswith('bdd100k_labels_images_val.json')]
    for n in tqdm(members, desc='bdd'):
        if not os.path.exists(f"{BDD_LOCAL}/{n}"):
            z.extract(n, BDD_LOCAL)

BDD_IMAGES_DIR = f"{BDD_LOCAL}/{os.path.dirname(next(n for n in members if n.endswith('.jpg')))}"
BDD_LABELS = f"{BDD_LOCAL}/{next(n for n in members if n.endswith('.json'))}"
print(BDD_IMAGES_DIR, len(os.listdir(BDD_IMAGES_DIR)), 'images')
print(BDD_LABELS, os.path.exists(BDD_LABELS))

bdd:   0%|          | 0/10001 [00:00<?, ?it/s]

/content/bdd/bdd100k/bdd100k/images/100k/val 10000 images
/content/bdd/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json True


In [ ]:
for f in sorted(os.listdir(CULANE_DRIVE_DIR), key=lambda f: ('annotations_new' in f, f)):
    flag = f"{CULANE_DIR}/.{f}.done"
    if f.endswith(('.tar.gz', '.tar')) and not os.path.exists(flag):
        print('extracting', f)
        subprocess.run(['tar', '-xf', f"{CULANE_DRIVE_DIR}/{f}", '-C', CULANE_DIR], check=True)
        open(flag, 'w').close()
if os.path.exists(f"{CULANE_DIR}/annotations_new"):
    shutil.copytree(f"{CULANE_DIR}/annotations_new", CULANE_DIR, dirs_exist_ok=True)
print(os.listdir(CULANE_DIR))

In [6]:
def prep(img, ds):
	return cv2.cvtColor(cv2.resize(img[DS[ds]['crop']:], (W, H), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

def resample(x, y):
	order = np.argsort(y)
	x = x[order]
	y = y[order]
	rows = np.arange(H)
	out = np.full(H, np.nan, np.float32)
	inside = (rows >= y[0]) & (rows <= y[-1])
	out[inside] = np.interp(rows[inside], y, x)
	return out

def to_rows(points, ds):
	c = DS[ds]
	return resample(points[:, 0] * W / c['w'], (points[:, 1] - c['crop']) * H / (c['h'] - c['crop']))

def pick_ego(lanes, min_row=H // 2):
	left, right = (None, -1e9), (None, 1e9)
	for x in lanes:
		rows = np.where(np.isfinite(x))[0]
		if len(rows) < 10 or rows[-1] < min_row:
			continue
		low = rows[-max(3, len(rows) // 3):]
		k, b = np.polyfit(low, x[low], 1)
		x_bottom = k * (H - 1) + b
		if x_bottom < W / 2 and x_bottom > left[1]:
			left = (x, x_bottom)
		if x_bottom >= W / 2 and x_bottom < right[1]:
			right = (x, x_bottom)
	empty = np.full(H, np.nan, np.float32)
	result = []
	for lane, _ in (left, right):
		if lane is None:
			result.append(empty)
		else:
			result.append(lane)
	return np.stack(result)

def culane_lanes(image_path):
	p = image_path[:-4] + '.lines.txt'
	out = []
	if not os.path.exists(p):
		return out
	for line in open(p):
		v = np.array(line.split(), np.float32).reshape(-1, 2)
		v = v[(v[:, 0] >= 0) & (v[:, 0] < DS['culane']['w'])]
		if len(v) >= 2:
			out.append(to_rows(v, 'culane'))
	return out

def merge_close(lanes, max_gap=10):
	merged = []
	for x in sorted(lanes, key=lambda x: -np.isfinite(x).sum()):
		joined = False
		for m in merged:
			both = np.isfinite(x) & np.isfinite(m)
			if both.sum() >= 3 and np.median(np.abs(x[both] - m[both])) < max_gap:
				fill = np.isfinite(x) & ~np.isfinite(m)
				m[fill] = x[fill]
				joined = True
				break
		if not joined:
			merged.append(x.copy())
	return merged

def bdd_lanes(entry):
	out = []
	for label in entry.get('labels') or []:
		a = label.get('attributes') or {}
		kind = str(a.get('laneType', label.get('category', '')))
		if a.get('laneDirection') != 'parallel' or 'crosswalk' in kind or 'curb' in kind:
			continue
		for poly in label.get('poly2d') or []:
			v = np.array(poly['vertices'], np.float32)
			if len(v) >= 2 and np.ptp(v[:, 1]) > 20:
				out.append(to_rows(v, 'bdd'))
	return merge_close(out)

def ego_lanes(path, ds):
	if ds == 'culane':
		return pick_ego(culane_lanes(path))
	return pick_ego(bdd_lanes(bdd_json[os.path.basename(path)]), int(0.3 * H))

In [7]:
def culane_list(f, condition):
	rel = [l.split()[0].lstrip('/') for l in open(f) if l.strip()]
	df = pd.DataFrame({'path': [f"{CULANE_DIR}/{r}" for r in rel], 'condition': condition, 'clip_id': [os.path.dirname(r) for r in rel]})
	return df[df.path.map(os.path.exists)]

def spread(df, n):
	df = df.sample(frac=1, random_state=SEED)
	df = df.assign(r=df.groupby('clip_id').cumcount()).sort_values('r', kind='stable')
	return df.head(n or len(df)).drop(columns='r')

def per_condition(df, n):
	n = min(n or len(df), df.condition.value_counts().min())
	parts = []
	for _, group in df.groupby('condition'):
		parts.append(spread(group, n))
	return pd.concat(parts)

def bdd_condition(a):
	if a.get('weather') in ('rainy', 'snowy', 'foggy'):
		return 'bad_weather'
	if a.get('timeofday') in ('night', 'dawn/dusk'):
		return a['timeofday'].replace('/', '_')
	if a.get('scene') == 'highway':
		return 'day_highway'
	return 'day_city'

train = culane_list(f"{CULANE_DIR}/list/train_gt.txt", 'unknown')
val = culane_list(f"{CULANE_DIR}/list/val_gt.txt", 'unknown')
val = val[~val.clip_id.isin(train.clip_id)]
test_dir = f"{CULANE_DIR}/list/test_split"
test = pd.concat([culane_list(f"{test_dir}/{f}", f[:-4].split('_', 1)[1]) for f in sorted(os.listdir(test_dir))])

bdd_images = set(os.listdir(BDD_IMAGES_DIR))
bdd_json = {e['name']: e for e in json.load(open(BDD_LABELS)) if e['name'] in bdd_images}
bdd = pd.DataFrame([{'path': f"{BDD_IMAGES_DIR}/{n}", 'condition': bdd_condition(e.get('attributes') or {}), 'clip_id': n[:-4]} for n, e in bdd_json.items()])

splits = {'train': spread(train, N_TRAIN), 'val': spread(val, N_VAL),
          'test_culane': per_condition(test, N_TEST_CULANE), 'test_bdd': per_condition(bdd, N_TEST_BDD)}
print('available: train', len(train), '| val', len(val))
for name, df in splits.items():
	print(name, len(df))

available: train 88880 | val 9675
train 45000
val 9500
test_culane 3798
test_bdd 3240


In [8]:
def build(name, df, ds):
	if os.path.exists(f"{PROCESSED_DIR}/{name}/meta.csv"):
		print(name, 'already on Drive')
		return
	out = f"{LOCAL_PROCESSED}/{name}"
	os.makedirs(out, exist_ok=True)
	df = df.reset_index(drop=True)
	images = np.lib.format.open_memmap(f"{out}/images.npy", 'w+', np.uint8, (len(df), H, W, 3))
	lanes = np.full((len(df), 2, H), np.nan, np.float32)

	def work(i):
		images[i] = prep(cv2.imread(df.path[i]), ds)
		lanes[i] = ego_lanes(df.path[i], ds)

	with ThreadPoolExecutor(16) as ex:
		list(tqdm(ex.map(work, range(len(df))), total=len(df), desc=name))
	images.flush()
	np.save(f"{out}/lanes.npy", lanes)
	meta = df.assign(path=df.path.str.replace(f"{CULANE_DIR}/", '').str.replace(f"{DATA_DIR}/", ''), dataset=ds,
	                 left=np.isfinite(lanes[:, 0]).any(1), right=np.isfinite(lanes[:, 1]).any(1))
	shutil.copytree(out, f"{PROCESSED_DIR}/{name}", dirs_exist_ok=True)
	for d in (out, f"{PROCESSED_DIR}/{name}"):
		meta.to_csv(f"{d}/meta.csv", index=False)
	meta[['path', 'condition', 'clip_id']].to_csv(f"{SPLITS_DIR}/{name}.csv", index=False)
	print(f"{name}: {len(df)} frames | left {meta.left.mean():.0%} | right {meta.right.mean():.0%}")

In [ ]:
def draw(img, lanes):
    img = np.array(img)
    for x, color in zip(lanes, [(0, 255, 0), (255, 0, 0)]):
        m = np.isfinite(x)
        if m.sum() > 1:
            cv2.polylines(img, [np.stack([x[m], np.arange(H)[m]], 1).astype(np.int32)], False, color, 2)
    return img

def show(name, n=9):
    d = f"{LOCAL_PROCESSED}/{name}"
    images, lanes, meta = np.load(f"{d}/images.npy", mmap_mode='r'), np.load(f"{d}/lanes.npy"), pd.read_csv(f"{d}/meta.csv")
    plt.figure(figsize=(18, 7))
    for j, i in enumerate(np.random.choice(len(meta), n, replace=False)):
        plt.subplot(3, 3, j + 1)
        plt.imshow(draw(images[i], lanes[i]))
        plt.title(meta.condition[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show('test_culane')
show('test_bdd')

In [ ]:
build('train', splits['train'], 'culane')

In [ ]:
def load(name):
    src, dst = f"{PROCESSED_DIR}/{name}", f"{LOCAL_PROCESSED}/{name}"
    if not os.path.exists(f"{dst}/meta.csv"):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    return np.load(f"{dst}/images.npy", mmap_mode='r'), np.load(f"{dst}/lanes.npy"), pd.read_csv(f"{dst}/meta.csv")

images, lanes, meta = load('val')
print(images.shape, lanes.shape)

In [9]:
remaining = train[~train.path.isin(splits['train'].path)].reset_index(drop=True)
print(len(remaining), 'extra training frames')
build('train_extra', remaining, 'culane')

43880 extra training frames


train_extra:   0%|          | 0/43880 [00:00<?, ?it/s]

train_extra: 43880 frames | left 88% | right 87%


In [10]:
drive.flush_and_unmount()
from google.colab import runtime
runtime.unassign()